# LaLonde (1986) / Dehejia–Wahba (1999) — NSW + PSID

**Paper:** Dehejia, R. & Wahba, S. (1999). *Causal Effects in Nonexperimental Studies.* JASA 94(448), 1053–1062. (bib key `dehejia1999causal`; LaLonde 1986: AER 76(4))

**Design:** observational ATT recovery vs an experimental benchmark. **Data:** real R `MatchIt::lalonde` extract (`lalonde_matchit.csv`, n=614: 185 NSW treated + 429 PSID-1 controls).

**What we reproduce:** naive OLS shows selection bias (negative); covariate-adjusted OLS and 1:1 propensity-score matching recover a positive ATT near the DW (1999) experimental benchmark of ≈ $1,794.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')  # headless-safe (notebooks run under nbclient in CI)
import matplotlib.pyplot as plt
import numpy as np
import statspai as sp
print('statspai', sp.__version__)

In [ ]:
df, _ = sp.replicate('lalonde_1986')
print(df.shape)
df.head()

In [ ]:
covs = ['age', 'educ', 'black', 'hispanic', 'married',
        'nodegree', 're74', 're75']
naive = sp.regress('re78 ~ treat', data=df, robust='hc1')
adj = sp.regress('re78 ~ treat + ' + ' + '.join(covs),
                 data=df, robust='hc1')
psm = sp.match(data=df, y='re78', treat='treat',
               covariates=covs, method='nearest')
naive_att = float(naive.params['treat'])
adj_att = float(adj.params['treat'])
psm_att = float(psm.estimate)
print(f'Naive OLS ATT     : {naive_att:8.1f}')
print(f'Adjusted OLS ATT  : {adj_att:8.1f}')
print(f'1:1 NN PSM ATT    : {psm_att:8.1f}')

In [ ]:
import pandas as pd
tab = pd.DataFrame([
    ['Naive OLS ATT ($)', naive_att, 'selection bias (negative)'],
    ['Adjusted OLS ATT ($)', adj_att, 'controls -> positive'],
    ['1:1 NN PSM ATT ($)', psm_att, 'recovers experimental ~$1794'],
], columns=['quantity', 'StatsPAI', 'note'])
tab

In [ ]:
# Figure: estimators vs the DW experimental benchmark
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Naive OLS', 'Adjusted OLS', '1:1 NN PSM'],
       [naive_att, adj_att, psm_att],
       color=['#d7301f', '#fdae61', '#2c7fb8'])
ax.axhline(1794, color='k', ls='--', label='DW (1999) experimental $1794')
ax.set_ylabel('ATT on 1978 earnings ($)')
ax.set_title('LaLonde / DW: recovering the experimental benchmark')
ax.legend(); fig.tight_layout(); fig

In [ ]:
# --- DRIFT GUARD ---
# Naive OLS and adjusted OLS reproduce to the dollar (R MatchIt parity).
assert abs(naive_att - (-635.0)) < 5.0, naive_att
assert abs(adj_att - 1548.2) < 5.0, adj_att
# 1:1 NN PSM: matching on binary covariates has tie-break sensitivity,
# so we guard the *scientific* claim (recovers the experimental
# benchmark, far above the biased naive estimate) rather than a
# brittle dollar pin. Current deterministic value ~ $1963.
assert naive_att < 0 < adj_att, (naive_att, adj_att)
assert 1500.0 < psm_att < 2500.0, psm_att
assert psm_att > naive_att + 2000  # matching removes the selection bias
print(f'OK: LaLonde reproduced (naive={naive_att:.0f}, '
      f'adj={adj_att:.0f}, PSM={psm_att:.0f}; benchmark ~$1794).')

**Result.** Naive OLS gives a *negative* ATT (−$635) on this PSID-control subset — the selection bias LaLonde flagged. Covariate adjustment (+$1,548) and 1:1 nearest-neighbour PSM (≈ +$1,963) both recover a positive effect near the DW (1999) experimental benchmark of ≈ $1,794. The PSM point is sensitive to tie-breaking on the binary covariates, so the guard targets the robust scientific conclusion. Doubly-robust DML and entropy balancing (modern track) are in `sp.replicate('lalonde_1986')`.